# KRYPTOS

This notebook demonstrates a few key techniques to determine the nature of K4, via the Vigenere cipher, which rely on cribs to evaluate solutions and reconstruct keys

In [1]:
alphabet = 'kryptosabcdefghijlmnquvwxz'
k4 = 'OBKRUOXOGHULBSOLIFBBWFLRVQQPRNGKSSOTWTQSJQSSEKZZWATJKLUDIAWINFBNYPVTTMZFPKWGDKZXTJCDIGKUHUAUEKCAR'.lower()

In [2]:
def vigenere_dec(alphabet = 'abcdefghijklmnopqrstuvwxyz', dec_key = None, input_string = None):

    dec_string = ""

    # Takes encrpytion key from user
    if dec_key == None:
        dec_key = input("Please enter encryption key: ")
        dec_key = dec_key.lower()

    # Takes string from user
    if input_string == None:
        input_string = input("Please enter a string of text: ")
        input_string = input_string.lower()

    # Lengths of input_string
    string_length = len(input_string)

    # Expands the encryption key to make it longer than the inputted string
    expanded_key = dec_key
    expanded_key_length = len(expanded_key)

    while expanded_key_length < string_length:
        # Adds another repetition of the encryption key
        expanded_key = expanded_key + dec_key
        expanded_key_length = len(expanded_key)

    key_position = 0

    for letter in input_string:
        if letter in alphabet:
            # cycles through each letter to find it's numeric position in the alphabet
            position = alphabet.find(letter)
            # moves along key and finds the characters value
            key_character = expanded_key[key_position]
            key_character_position = alphabet.find(key_character)
            key_position = key_position + 1
            # changes the original of the input string character
            new_position = position - key_character_position
            new_position = new_position % 26
            if new_position >= len(alphabet) or new_position < 0:
                print(f'pos {new_position}\n')
            new_character = alphabet[new_position]
            dec_string = dec_string + new_character
        else:
            dec_string = dec_string + letter
    return(dec_string)

# Cribs

Cribs have been provided, and can be used to deduce whether a recovered text contains the cribs, and therefore is the correct plaintext

In [3]:
conditions_1 = [(63,'b'),(64,'e'),(65,'r'),(66,'l'),(67,'i'),(68,'n')]
conditions_2 = [(69,'c'),(70,'l'),(71,'o'),(72,'c'),(73,'k')]
conditions_3 = [(25,'n'),(26,'o'),(27,'r'),(28,'t'),(29,'h'),(30,'e'),(31,'a'),(32,'s'),(33,'t')]
conditions_4 = [(21,'e'),(22,'a'),(23,'s'),(24,'t')]

conditions = sorted(conditions_1 + conditions_2 + conditions_3 + conditions_4, key= lambda i : i[0])
conditions

[(21, 'e'),
 (22, 'a'),
 (23, 's'),
 (24, 't'),
 (25, 'n'),
 (26, 'o'),
 (27, 'r'),
 (28, 't'),
 (29, 'h'),
 (30, 'e'),
 (31, 'a'),
 (32, 's'),
 (33, 't'),
 (63, 'b'),
 (64, 'e'),
 (65, 'r'),
 (66, 'l'),
 (67, 'i'),
 (68, 'n'),
 (69, 'c'),
 (70, 'l'),
 (71, 'o'),
 (72, 'c'),
 (73, 'k')]

## Demonstrating the length of K4 keyphrase, when encrypted as a single pass vigenere encryption

We can deduce possible lengths of the keyphrase, by attempting iteratively creating a keyword, then modifying a single letter in the keyphrase until the letter correctly reveals the crib letters shown above.

For this process, let us assume that any n letter keyphrase which only contains the first letter of the alphabet ('k' for us), results in the decrypted text being the same as the original cryptext.

For example, a key of five letters 'kkkkk', result with the decrypted text being the same as the cryptext.

In [4]:
vigenere_dec(alphabet, 'kkkkk',k4)

'obkruoxoghulbsolifbbwflrvqqprngkssotwtqsjqssekzzwatjkludiawinfbnypvttmzfpkwgdkzxtjcdigkuhuauekcar'

Using this fact, allows us to quickly construct keys which fulfil the cribs.

Below, the recovered keys will contain the letter 'k' and can be considered a  wildcard character. 

In [5]:
for key_length in range(1,len(k4) + 1):
  key = 'k' * key_length
  key_modifications = []
  key_failed = False

  for c in conditions:
    if key_failed:
      break

    condition_key_position = c[0] % key_length + 1

    if condition_key_position in key_modifications:
      key_failed = True
      print(f'{key_length} failed, on condition ({c[0]},{c[1]})')
      break

    for j in range(1,26):
      key_test = key[:condition_key_position-1] + alphabet[j % 26] + key[condition_key_position:]
      # print(f'{key_test} training on condition ({c[0]},{c[1]})')

      if vigenere_dec(alphabet, key_test, k4)[c[0]] == c[1]:
        key = key_test
        key_modifications.append(condition_key_position)
        # print(f'key length {key_length} trained, on condition ({c[0]},{c[1]})')
        break
  if not key_failed:
    plaintext = vigenere_dec(alphabet, key, k4)
    plaintext = ''.join([plaintext[i].upper() if i % key_length + 1 in key_modifications else plaintext[i].lower() for i in range(len(plaintext))])
    key = ''.join([key[i].upper() if i % key_length + 1 in key_modifications else key[i].lower() for i in range(len(key))])
    
    print(f'{key_length} {key} {plaintext}')

1 failed, on condition (22,a)
2 failed, on condition (23,s)
3 failed, on condition (24,t)
4 failed, on condition (25,n)
5 failed, on condition (26,o)
6 failed, on condition (27,r)
7 failed, on condition (28,t)
8 failed, on condition (29,h)
9 failed, on condition (30,e)
10 failed, on condition (31,a)
11 failed, on condition (32,s)
12 failed, on condition (33,t)
13 failed, on condition (63,b)
14 failed, on condition (63,b)
15 failed, on condition (63,b)
16 failed, on condition (63,b)
17 failed, on condition (63,b)
18 failed, on condition (63,b)
19 failed, on condition (63,b)
20 failed, on condition (63,b)
21 failed, on condition (63,b)
22 failed, on condition (65,r)
23 failed, on condition (67,i)
24 failed, on condition (69,c)
25 failed, on condition (71,o)
26 failed, on condition (73,k)
27 YWOYNkYkkELYOIECBAQkkRDUMRI PEUZYoVogPTIPLQBAOHbwEASTNORTHEAsTotFGMRRCWXTSzzVWCXZYNGDOTiLfbBERLINCLOCkwFKOAWIHFOGQkNhuVTCUQVM
28 WOYNkYkELYOIECBAQkkkkRDUMRIY BPXBuPxQVFJYWWWDUfbbwEASTNORTHEAsToNSYILOE

So, we can see that keys of length 27-29 and 53-97 can be used to encrypt using vingenere and recover the crib text, however keys of length 1-26 and 30-52 do not recover the crib text.

Note the partial plain text recovered using the recovered keys uses an UPPERCASE letter to show due to a key letter being modified, the recovered plain text was impacted.

**Possible extensions include:**
- Removal of some of the cribs to test for a smaller key length
- Addition of some test cribs to experiment the impact of self defined cribs 

## Demonstration of double encryption recovery

It is possible to 'quickly' determine if two words can be used to decrypt 
a crypt text, by first constructing a lookup table which describes the
result of encrypting a plain text character, using two other characters (one at a time).

The example below uses an excerpt from Alice in Wonderland with the same length as K4, and some fake conditions which occur at the same locations as the cribs for K4

In [6]:
plain_text = 'ohyoucanthelpthatsaidthecatwereallmadhereimmadyouremadhowdoyouknowimmadyoumustbesaidthecatabcdefg'
key_word_1 = 'mushroom'
key_word_2 = 'drinkme'
conditions = [(21, 't'),
              (22, 'h'),
              (23, 'e'),
              (24, 'c'),
              (25, 'a'),
              (26, 't'),
              (27, 'w'),
              (28, 'e'),
              (29, 'r'),
              (30, 'e'),
              (31, 'a'),
              (32, 'l'),
              (33, 'l'),
              (63, 'n'),
              (64, 'o'),
              (65, 'w'),
              (66, 'i'),
              (67, 'm'),
              (68, 'm'),
              (69, 'a'),
              (70, 'd'),
              (71, 'y'),
              (72, 'o'),
              (73, 'u')]

In [7]:
def vigenere_enc(alphabet = 'abcdefghijklmnopqrstuvwxyz', enc_key = None, input_string = None):
    assert len(alphabet) == 26, 'Expected 26 characters'
    enc_string = ""

    # Takes encrpytion key from user
    if enc_key is None: 
        enc_key = input("Please enter encryption key: ")
        enc_key = enc_key.lower()

    # Takes string from user
    if input_string is None:
        input_string = input("Please enter a string of text: ")
        input_string = input_string.lower()

    # Lengths of input_string
    string_length = len(input_string)

    # Expands the encryption key to make it longer than the inputted string
    expanded_key = enc_key
    expanded_key_length = len(expanded_key)

    while expanded_key_length < string_length:
        # Adds another repetition of the encryption key
        expanded_key = expanded_key + enc_key
        expanded_key_length = len(expanded_key)

    key_position = 0

    for letter in input_string:
        if letter in alphabet:
            # cycles through each letter to find it's numeric position in the alphabet
            position = alphabet.find(letter)
            # moves along key and finds the characters value
            key_character = expanded_key[key_position]
            key_character_position = alphabet.find(key_character)
            key_position = key_position + 1
            # changes the original of the input string character
            new_position = position + key_character_position
            if new_position >= 26:
                new_position = new_position - 26
            #print(new_position)
            new_character = alphabet[new_position]
            enc_string = enc_string + new_character
        else:
            enc_string = enc_string + letter
    return(enc_string)

Let's construct a fake cryptext to demonstrate encryption, which will be used to demonstrate decryption below

In [8]:
pass_1 = vigenere_enc(alphabet, key_word_1, plain_text)
crypt_text = vigenere_enc(alphabet, key_word_2, pass_1)
crypt_text

'adwfvswuwxdovqpkeqguvnqmqyyvvaomctcofbcnuubawbaixsmuriebzskcsmjuxahsewzufcxrmnhmlygciqoqzlxsetctw'

Below, we construct a lookup table, which contains a dictionary of plain text, key 1 and key 2 characters. 

In [9]:
dk = {}
for i in alphabet:
    for j in alphabet:
        for k in alphabet:

            m = vigenere_dec(alphabet, j, i)
            n = vigenere_dec(alphabet, k, m)

            if i not in dk.keys():
              dk[i] = {}
            dk0 = dk[i]
            
            if n not in dk0.keys():
              dk0[n] = {}
            dk1 = dk0[n]

            dk1[k] = j

In [10]:
def check(crypt_text, condtions, w0, w1):
    l = len(crypt_text)

    # create pseudo keys, which repeat the key 
    # over and over ie. rubarbrubarbrubarb
    # for the length of the crypt_text  
    k0 = (w0 * l)[:l]
    k1 = (w1 * l)[:l]

    for c in conditions:
      p = c[0]

      # Is the condition satisfied? 
      if dk[crypt_text[p]][k0[p]][k1[p]] != c[1]:
        return False

    print(f'{w0}, {w1}')

    return True

In [11]:
check(crypt_text, conditions, 'deduced','somewhere')

False

In [12]:
check(crypt_text, conditions, 'mushroom','drinkme')

mushroom, drinkme


True

Interestingly, Vingenere provides a simple commutative relationship, as it doesn't matter which order keywords are used to encrypt with

In [13]:
check(crypt_text, conditions, 'drinkme','mushroom')

drinkme, mushroom


True

Using this process, a brute force test of all combinations of the English dictionary were tested.  No two words could be used to satisfy the original K4 Kryptos cryptext and cribs.

**Possible extensions include:**
 - Brute force exploration using multiple non-english characters
 - Extension to test for triple encryption 